# Opis

Krótki notebook, który pozwala przetestować działanie różnych elementów implementacyjnych w szybki sposób.

# Importy

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
import yaml
import sys
import tqdm
import wandb
import json
sys.path.append('../') # Dodajemy katalog wyżej, żeby src był widoczny
from src.models.KlejdaGAE.GraphAutoencoder import GraphAutoencoder
from src.models.KlejdaGAE.VariationalGraphAutoencoder import VariationalGraphAutoencoder
from utils.FramsticksGraphDataset import FramsticksGraphDataset

W0604 00:03:05.430000 13024 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Przetwarzanie

## Pomocnicza klasa tworząca sztuczne przykłady osobników

In [2]:
class FramsticksDummyDataset(Dataset):
    """
    Tworzy zbiór losowych grafów o stałym rozmiarze, symulujących dane z Framsticks.
    Do weryfikacji czy model działa, uczy się i nie pojawiają się błędy.
    """
    def __init__(self, num_samples=1000, max_nodes=15, in_channels=3):
        self.num_samples = num_samples
        self.max_nodes = max_nodes
        self.in_channels = in_channels

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
		# Tworzy macierz max_nodes x in_channels z wartościami w zakresie -1 do 1
        x = torch.rand((self.max_nodes, self.in_channels)) * 2 - 1.0

        # Generuje losową, symetryczną macierz sąsiedztwa A z zerami i jedynkami
        adj = torch.rand((self.max_nodes, self.max_nodes))
		# Sztuczka, aby zapewnić symetrię
        adj = (adj + adj.T) / 2
		# Progowanie na 0 i 1 korzystając z odcięcia (większe odcięcie, rzadsza struktura
        adj = (adj > 0.7).float()
		# Wypełnienie diagonali
        adj.fill_diagonal_(1.0)

        return x, adj




## Załadowanie danych i przygotowanie do przetwarzania

In [3]:
# dataset = FramsticksDummyDataset(num_samples=1000)

with open("../results/sampled_individuals.json", "r") as f:
	genotypes_json = json.load(f)

genotypes = []
for el in genotypes_json:
	genotypes.append(el["genotype"])

with open("../configs/klejda_gae_config.yaml") as f:
    config = yaml.safe_load(f)

dataset = FramsticksGraphDataset(genotypes,config["max_nodes"])
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\witek\_netrc.
wandb: Currently logged in as: witekadrian7 (witekadrian7-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## GAE

In [6]:
modelGAE = GraphAutoencoder(config)
wandb_logger = WandbLogger(project="Framsticks-GAE", name="GAE-Baseline-Test", save_dir = config["save_dir"])

trainer = pl.Trainer(
    max_epochs=60,
    logger=wandb_logger,
    log_every_n_steps=5,
    accelerator="auto",
    devices=1
)
trainer.fit(modelGAE, train_dataloaders=dataloader)
wandb.finish()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
wandb: setting up run fj6basaz
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in checkpoints\wandb\run-20260604_000500-fj6basaz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run GAE-Baseline-Test
wandb:  View project at https://wandb.ai/witekadrian7-none/Framsticks-GAE
wandb:  View run at https://wandb.ai/witekadrian7-none/Framsticks-GAE/runs/fj6basaz
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type     | Param

Epoch 59: 100%|██████████| 159/159 [00:03<00:00, 48.83it/s, v_num=asaz, train/loss_total=0.192]

`Trainer.fit` stopped: `max_epochs=60` reached.


Epoch 59: 100%|██████████| 159/159 [00:03<00:00, 48.51it/s, v_num=asaz, train/loss_total=0.192]


wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:               epoch ▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
wandb:        train/loss_A █▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:        train/loss_X █▄▃▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:    train/loss_total █▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: trainer/global_step ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
wandb: 
wandb: Run summary:
wandb:               epoch 59
wandb:        train/loss_A 0.00017
wandb:        train/loss_X 0.02046
wandb:    train/loss_total 0.19168
wandb: trainer/global_step 9539
wandb: 
wandb:  View run GAE-Baseline-Test at: https://wandb.ai/witekadrian7-none/Framsticks-GAE/runs/fj6basaz
wandb:  View project at: https://wandb.ai/witekadrian7-none/Framsticks-GAE
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: checkpoints\wandb\run-20260604_000500-fj6basaz\logs


## VGAE

In [5]:
modelVGAE = VariationalGraphAutoencoder(config)

wandb_logger = WandbLogger(project="Framsticks-VGAE", name="VGAE-Baseline-Test", save_dir = config["save_dir"])
trainer = pl.Trainer(
    max_epochs=10,
    logger=wandb_logger,
    log_every_n_steps=5,
    accelerator="auto",
    devices=1
)

trainer.fit(modelVGAE, train_dataloaders=dataloader)
wandb.finish()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\pytorch_lightning\trainer\configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
wandb: setting up run 9itv7mf9
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in checkpoints\wandb\run-20260604_000337-9itv7mf9
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run VGAE-Baseline-Test
wandb:  View project at https://wan

Epoch 9: 100%|██████████| 159/159 [00:02<00:00, 58.63it/s, v_num=7mf9]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 159/159 [00:02<00:00, 58.18it/s, v_num=7mf9]


wandb: updating run metadata
wandb: uploading history steps 6-9, summary
wandb: 
wandb: Run history:
wandb:               epoch ▁▂▃▃▄▅▆▆▇█
wandb:        train/loss_A █▂▂▁▁▁▁▁▁▁
wandb:       train/loss_KL █▆▄▂▃▅▂▃▂▁
wandb:        train/loss_X █▄▂▂▁▁▁▁▁▁
wandb:    train/loss_total █▂▂▁▁▁▁▁▁▁
wandb: trainer/global_step ▁▂▃▃▄▅▆▆▇█
wandb: 
wandb: Run summary:
wandb:               epoch 9
wandb:        train/loss_A 0.00087
wandb:       train/loss_KL 13.42511
wandb:        train/loss_X 0.03384
wandb:    train/loss_total 0.97316
wandb: trainer/global_step 1589
wandb: 
wandb:  View run VGAE-Baseline-Test at: https://wandb.ai/witekadrian7-none/Framsticks-VGAE/runs/9itv7mf9
wandb:  View project at: https://wandb.ai/witekadrian7-none/Framsticks-VGAE
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: checkpoints\wandb\run-20260604_000337-9itv7mf9\logs
